# Templates Directory Management Notebook

This notebook provides utilities to:
1. Create a directory listing of templates
2. Save the listing to a Markdown file
3. Update base.html with template links
4. Modify server.py to serve these templates dynamically

## Import Required Libraries

In [ ]:
# Import libraries for directory operations and file handling
import os
import sys
import re
import datetime
import json
from pathlib import Path

## List Files in Directory

We'll scan the templates directory and gather information about all template files.

In [ ]:
# Define the templates directory path
templates_dir = Path('src/web/templates')

# Check if directory exists
if not templates_dir.exists():
    print(f"Directory {templates_dir} does not exist.")
    print("Creating directory structure...")
    
    # Create the directory if it doesn't exist
    templates_dir.mkdir(parents=True, exist_ok=True)
    print(f"Created directory: {templates_dir}")
    
# List all files in the templates directory
template_files = []

try:
    # Get all HTML files in the templates directory
    template_files = list(templates_dir.glob('*.html'))
    
    # Display results
    print(f"Found {len(template_files)} template files:")
    for template_file in template_files:
        print(f"- {template_file.name}")
        
except Exception as e:
    print(f"Error listing template files: {e}")

In [ ]:
# Collect additional metadata about each template file
template_metadata = []

for template_file in template_files:
    try:
        # Get file stats
        stats = template_file.stat()
        
        # Read first few lines to extract title if available
        title = template_file.name
        description = ""
        
        with open(template_file, 'r', encoding='utf-8') as f:
            content = f.read(1024)  # Read just the beginning
            
            # Try to extract title from HTML
            title_match = re.search(r'<title>(.*?)</title>', content, re.IGNORECASE)
            if title_match:
                title = title_match.group(1)
                
            # Try to extract meta description
            desc_match = re.search(r'<meta\s+name="description"\s+content="(.*?)"\s*/?>', content, re.IGNORECASE)
            if desc_match:
                description = desc_match.group(1)
        
        template_metadata.append({
            'filename': template_file.name,
            'title': title,
            'description': description,
            'modified': datetime.datetime.fromtimestamp(stats.st_mtime).strftime('%Y-%m-%d %H:%M:%S'),
            'size': f"{stats.st_size / 1024:.1f} KB"
        })
        
    except Exception as e:
        print(f"Error processing {template_file.name}: {e}")

# Display the collected metadata
print("\nTemplate Metadata:")
for item in template_metadata:
    print(f"\n{item['filename']}:")
    print(f"  Title: {item['title']}")
    print(f"  Description: {item['description']}")
    print(f"  Last Modified: {item['modified']}")
    print(f"  Size: {item['size']}")

## Write Directory Listing to Markdown File

Now we'll create a Markdown file with the template listing and metadata.

In [ ]:
# Define the output markdown file
output_md_path = Path('src/web/templates_listing.md')

try:
    with open(output_md_path, 'w', encoding='utf-8') as md_file:
        # Write header
        md_file.write("# Templates Directory Listing\n\n")
        md_file.write(f"Generated on: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        
        # Write table header
        md_file.write("| Filename | Title | Description | Last Modified | Size |\n")
        md_file.write("|----------|-------|-------------|--------------|------|\n")
        
        # Write table rows
        for item in template_metadata:
            md_file.write(f"| [{item['filename']}](templates/{item['filename']}) | {item['title']} | {item['description']} | {item['modified']} | {item['size']} |\n")
        
        # Additional information
        md_file.write("\n\n## Usage Instructions\n\n")
        md_file.write("To use these templates in your application:\n\n")
        md_file.write("1. Import the template in your route handler\n")
        md_file.write("2. Render the template with the required context variables\n")
        md_file.write("3. Make sure to update the base.html if you're using template inheritance\n")
    
    print(f"Successfully wrote template listing to {output_md_path}")
    
except Exception as e:
    print(f"Error writing to Markdown file: {e}")

## Update base.html

Now we'll demonstrate how to update the base.html file to include links to all available templates.

In [ ]:
# Define the base.html path
base_html_path = templates_dir / 'base.html'

# Check if base.html exists, create a sample one if it doesn't
if not base_html_path.exists():
    print(f"Base template {base_html_path} does not exist. Creating a sample one...")
    
    with open(base_html_path, 'w', encoding='utf-8') as f:
        f.write("""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{% block title %}Default Title{% endblock %}</title>
    <meta name="description" content="{% block description %}Default description{% endblock %}">
    <link rel="stylesheet" href="/static/css/styles.css">
    {% block extra_css %}{% endblock %}
</head>
<body>
    <header>
        <nav>
            <ul>
                <li><a href="/">Home</a></li>
                <!-- Template navigation will be inserted here -->
            </ul>
        </nav>
    </header>
    
    <main>
        {% block content %}{% endblock %}
    </main>
    
    <footer>
        <p>&copy; {% now 'Y' %} ImpressionCore</p>
    </footer>
    
    <script src="/static/js/main.js"></script>
    {% block extra_js %}{% endblock %}
</body>
</html>
""")
    print(f"Created sample base.html template at {base_html_path}")

# Now update base.html with template links
try:
    # Read the current content
    with open(base_html_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Generate the navigation links
    nav_links = ""
    for item in template_metadata:
        # Skip base.html itself
        if item['filename'] == 'base.html':
            continue
            
        # Convert filename to route (remove .html extension)
        route_name = item['filename'].replace('.html', '')
        if route_name == 'index':
            route = '/'
        else:
            route = f'/{route_name}'
            
        nav_links += f'                <li><a href="{route}">{item["title"]}</a></li>\n'
    
    # Replace the navigation placeholder
    updated_content = re.sub(
        r'(<!-- Template navigation will be inserted here -->)',
        f'<!-- Template navigation will be inserted here -->\n{nav_links}                <!-- End of template navigation -->',
        content
    )
    
    # Write the updated content
    with open(base_html_path, 'w', encoding='utf-8') as f:
        f.write(updated_content)
        
    print(f"Successfully updated {base_html_path} with template navigation links")
    
except Exception as e:
    print(f"Error updating base.html: {e}")

## Update server.py

Finally, let's modify server.py to dynamically serve the templates we've identified.

In [ ]:
# Define the server.py path
server_py_path = Path('src/web/server.py')

# Check if server.py exists, create a sample one if it doesn't
if not server_py_path.exists():
    print(f"Server file {server_py_path} does not exist. Creating a sample one...")
    
    # Create directory if it doesn't exist
    server_py_path.parent.mkdir(parents=True, exist_ok=True)
    
    with open(server_py_path, 'w', encoding='utf-8') as f:
        f.write("""from flask import Flask, render_template

app = Flask(__name__)

@app.route('/')
def index():
    return render_template('index.html')

# Additional routes will be inserted here

if __name__ == '__main__':
    app.run(debug=True)
""")
    print(f"Created sample server.py at {server_py_path}")

# Now update server.py with routes for all templates
try:
    # Read the current content
    with open(server_py_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Generate routes for each template
    new_routes = ""
    for item in template_metadata:
        # Skip base.html and index.html (index already has a route)
        if item['filename'] in ['base.html', 'index.html']:
            continue
            
        # Create route name from filename (remove .html extension)
        route_name = item['filename'].replace('.html', '')
        
        # Add the route
        new_routes += f"""
@app.route('/{route_name}')
def {route_name.replace('-', '_')}():
    return render_template('{item['filename']}')
"""
    
    # Replace the routes placeholder
    updated_content = re.sub(
        r'(# Additional routes will be inserted here)',
        f'# Additional routes will be inserted here\n{new_routes}# End of additional routes',
        content
    )
    
    # Write the updated content
    with open(server_py_path, 'w', encoding='utf-8') as f:
        f.write(updated_content)
        
    print(f"Successfully updated {server_py_path} with routes for all templates")
    
except Exception as e:
    print(f"Error updating server.py: {e}")

## Summary

This notebook has performed the following operations:

1. Imported the necessary libraries for directory operations and file handling
2. Listed and analyzed all HTML template files in the src/web/templates directory
3. Created a Markdown file (templates_listing.md) with details about the templates
4. Updated base.html to include navigation links to all templates
5. Modified server.py to include routes for all templates

These changes establish a foundation for managing web templates in the ImpressionCore project.

### Next Steps

1. Implement template inheritance using the base.html file
2. Add CSS styling to the templates
3. Set up form handling and data processing in the route handlers
4. Implement dynamic content loading based on user interactions